# Phase 2: Dual-Path Sequential Naive Bayes & Enriched Stacking Architecture
**Project**: AURA - Adaptive User Risk Analytics
**Dataset**: Open University Learning Analytics Dataset (OULAD)
**Target Execution**: Google Colab (CPU)

---

### Objectives
1. **Pipeline Verification**: Load raw OULAD data or ingest via Phase 1 pipeline.
2. **Cartesian Grid Reshaping**: Format dynamic clicks and milestone scores into (N, 40) tensors aligned with static demographics.
3. **Dual-Path Naive Bayes Training**: Fit class-conditional Gaussian distributions on the training split with evidence discounting (gamma = 0.15) to prevent autocorrelation overcounting.
4. **Milestone Weight Optimization**: Calibrate milestone temporal weights W using validation log-loss grid search.
5. **Trajectory Evaluation**: Compute sequential posterior distributions week-by-week with temporal freezing upon withdrawal.
6. **Enriched Stacking Meta-Classifier**: Train multinomial logistic regression fusing dynamic Bayesian posteriors, cumulative behavioral summaries, and standardized static demographics at cutoff weeks 4, 8, 12, and 39.
7. **Validation Benchmark**: Measure Macro F1, Per-Class Recall, One-vs-Rest ROC-AUC, and At-Risk (Withdrawn/Fail) Recall against Phase 1 baselines.

In [ ]:
# Install dependencies in Colab environment
!pip install -q kagglehub polars pyarrow scikit-learn seaborn matplotlib scipy lightgbm

import os
import sys
import shutil
import time
from pathlib import Path

if "/content" not in sys.path:
    sys.path.insert(0, "/content")
if "." not in sys.path:
    sys.path.append(".")

import polars as pl
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, recall_score, classification_report

from src.data_loader import run_pipeline, TARGET_MAP, RAW_SCHEMAS
from src.dual_path_nb import DualPathNaiveBayes, default_milestone_weight_array, calibrate_milestone_weights
from src.meta_classifier import MetaClassifierWrapper, stack_features, stack_enriched_features
from src.evaluation import evaluate_multiclass_predictions

pl.enable_string_cache()
np.random.seed(42)

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
FIG_DIR = Path("figures")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Environment and modules initialized successfully.")\n

In [ ]:
# Ingest raw OULAD tables via kagglehub if not present
import kagglehub

REQUIRED_TABLES = [
    "assessments.csv",
    "courses.csv",
    "studentAssessment.csv",
    "studentInfo.csv",
    "studentRegistration.csv",
    "studentVle.csv",
    "vle.csv",
]

missing_tables = [t for t in REQUIRED_TABLES if not (RAW_DIR / t).exists() or (RAW_DIR / t).stat().st_size == 0]

if missing_tables:
    print(f"Downloading OULAD dataset via kagglehub (missing: {missing_tables})...")
    slugs = [
        "anlgrbz/student-demographics-online-education-dataoulad",
        "rocki37/open-university-learning-analytics-dataset",
    ]
    download_dir = None
    for slug in slugs:
        try:
            print(f"Trying slug: {slug}...")
            download_dir = kagglehub.dataset_download(slug)
            print(f"Downloaded to: {download_dir}")
            break
        except Exception as e:
            print(f"Slug {slug} download failed: {e}")

    if download_dir and Path(download_dir).exists():
        for root, _, files in os.walk(download_dir):
            for f in files:
                for req in REQUIRED_TABLES:
                    if f.lower() == req.lower():
                        dest = RAW_DIR / req
                        shutil.copy2(os.path.join(root, f), dest)
                        print(f"Copied {f} -> {dest}")

# Verify all tables are present
for t in REQUIRED_TABLES:
    assert (RAW_DIR / t).exists(), f"Missing required raw table: {t}"
print("All raw OULAD tables present and verified.")

In [ ]:
# Run Phase 1 pipeline if processed artifacts do not already exist
req_processed = [
    "dynamic_features_weekly.parquet",
    "static_demographics_raw.parquet",
    "encoded_demographics.npy",
    "cohort_splits.npz",
]
missing_processed = [f for f in req_processed if not (PROCESSED_DIR / f).exists()]

if missing_processed:
    print(f"Executing Phase 1 ingestion pipeline for missing artifacts: {missing_processed}...")
    t0 = time.perf_counter()
    run_pipeline(RAW_DIR, PROCESSED_DIR)
    print(f"Phase 1 pipeline generated in {time.perf_counter() - t0:.2f}s")
else:
    print("Processed artifacts already exist. Skipping pipeline execution.")

In [ ]:
# Load processed artifacts
print("Loading processed artifacts into memory...")
dynamic_df = pl.read_parquet(PROCESSED_DIR / "dynamic_features_weekly.parquet")
base_students_df = pl.read_parquet(PROCESSED_DIR / "static_demographics_raw.parquet")
encoded_demographics = np.load(PROCESSED_DIR / "encoded_demographics.npy")
splits = np.load(PROCESSED_DIR / "cohort_splits.npz", allow_pickle=True)

train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
test_idx = splits["test_idx"]

n_students = base_students_df.shape[0]
print(f"Total unique students: {n_students:,}")
print(f"Train cohort: {len(train_idx):,} | Val cohort: {len(val_idx):,} | Test cohort: {len(test_idx):,}")
print(f"Encoded demographic matrix shape: {encoded_demographics.shape}")

In [ ]:
# Reshape Cartesian grid into aligned tensors (N, 40)
print("Pivoting dynamic Cartesian grid into (N, 40) tensors...")
t0 = time.perf_counter()

# Construct student index map from base_students_df
student_keys = base_students_df.select([
    "code_module", "code_presentation", "id_student"
]).with_row_index("student_idx")


granular_vle = pl.read_parquet(PROCESSED_DIR / "granular_vle.parquet")
vle_wide = granular_vle.pivot(
    on="activity_type",
    index=["code_module", "code_presentation", "id_student", "week"],
    values="raw_clicks",
    aggregate_function="sum"
).fill_null(0).sort(["code_module", "code_presentation", "id_student", "week"])

# Get activity columns
activity_cols = [c for c in vle_wide.columns if c not in ["code_module", "code_presentation", "id_student", "week"]]

# Join back with dynamic_df
joined_granular = dynamic_df.join(
    vle_wide, on=["code_module", "code_presentation", "id_student", "week"], how="left"
).fill_null(0).sort(["code_module", "code_presentation", "id_student", "week"])

joined_granular = joined_granular.join(
    student_keys, on=["code_module", "code_presentation", "id_student"], how="inner"
).sort(["student_idx", "week"])
\njoined_dynamic = joined_granular.clone()  # 
    student_keys, on=["code_module", "code_presentation", "id_student"], how="inner"
).sort(["student_idx", "week"])

# Pivot clicks and scores
clicks_pivot = joined_dynamic.pivot(on="week", index="student_idx", values="log_clicks").sort("student_idx")
scores_pivot = joined_dynamic.pivot(on="week", index="student_idx", values="normalized_score").sort("student_idx")

week_cols = [c for c in clicks_pivot.columns if c != "student_idx"]
week_cols = sorted(week_cols, key=lambda x: int(x))

X_clicks = clicks_pivot.select(week_cols).to_numpy().astype(np.float64)
X_scores = scores_pivot.select(week_cols).to_numpy().astype(np.float64)

# Extract unregistration weeks, student IDs, and labels
student_ids = base_students_df["id_student"].to_numpy()
unreg_dates = base_students_df["date_unregistration"].to_numpy()
unreg_weeks = np.where(
    np.isnan(unreg_dates.astype(float)),
    -1.0,
    np.where(unreg_dates < 0, 0.0, np.floor(unreg_dates / 7.0) + 1.0)
)


X_activities = {}
for act in activity_cols:
    act_pivot = joined_granular.pivot(on="week", index="student_idx", values=act).sort("student_idx")
    X_activities[act] = act_pivot.select(week_cols).to_numpy().astype(np.float64)

targets = base_students_df["final_result"].to_pandas().map(TARGET_MAP).to_numpy(dtype=np.int64)

print(f"X_clicks shape: {X_clicks.shape}")
print(f"X_scores shape: {X_scores.shape} (NaN count: {np.isnan(X_scores).sum():,})")
print(f"Tensors materialized in {time.perf_counter() - t0:.2f}s")

In [ ]:
# Fit Dual-Path Naive Bayes with Evidence Discounting (gamma = 0.15)
print("Fitting Dual-Path Naive Bayes model on training cohort with gamma=0.15...")
t0 = time.perf_counter()

X_clicks_train = X_clicks[train_idx]
X_scores_train = X_scores[train_idx]
y_train = targets[train_idx]

nb_model = DualPathNaiveBayes(num_classes=4, var_smoothing=1e-4, discount_factor=0.15)
nb_model.fit(X_clicks_train, X_scores_train, y_train)

print(f"Model fitted in {time.perf_counter() - t0:.3f}s")
print("Class empirical priors:")
inv_map = {v: k for k, v in TARGET_MAP.items()}
for k in range(4):
    print(f"  Class {k} ({inv_map[k]}): {nb_model.priors_[k]:.4f}")

In [ ]:
# Calibrate Milestone Weights on Validation Cohort
print("Calibrating milestone temporal weights W via validation grid search...")
t0 = time.perf_counter()

X_clicks_val = X_clicks[val_idx]
X_scores_val = X_scores[val_idx]
y_val = targets[val_idx]

grid_params = {
    "assessment_weeks": [3, 8, 19, 24],
    "base_weights": [0.5, 1.0],
    "milestone_weights": [1.0, 1.5, 2.0, 2.5],
    "eval_cutoff_week": 39,
}

best_weights, best_config = calibrate_milestone_weights(
    nb_model=nb_model,
    val_clicks=X_clicks_val,
    val_scores=X_scores_val,
    val_labels=y_val,
    grid_search_params=grid_params,
)

print(f"Grid search completed in {time.perf_counter() - t0:.2f}s")
print(f"Best configuration: {best_config}")

# Compare log-loss of Uniform (W=1) vs Calibrated Milestone Weights
uniform_weights = np.ones(40, dtype=np.float64)
preds_uniform = nb_model.predict_terminal_posteriors(X_clicks_val, X_scores_val, cutoff_week=39, weights=uniform_weights, discount_factor=0.15)
preds_calibrated = nb_model.predict_terminal_posteriors(X_clicks_val, X_scores_val, cutoff_week=39, weights=best_weights, discount_factor=0.15)

loss_uniform = -np.mean(np.log(np.clip(preds_uniform[np.arange(len(y_val)), y_val], 1e-15, 1.0)))
loss_calibrated = -np.mean(np.log(np.clip(preds_calibrated[np.arange(len(y_val)), y_val], 1e-15, 1.0)))

print(f"Validation Log-Loss: Uniform W=1: {loss_uniform:.4f} | Calibrated W: {loss_calibrated:.4f}")

In [ ]:
# Compute Sequential Posterior Trajectories Across All Weeks
print("Generating full posterior probability trajectories (N, 40, 4) with temporal withdrawal freezing...")
t0 = time.perf_counter()

all_posteriors = nb_model.predict_weekly_posteriors(
    weekly_clicks=X_clicks,
    weekly_scores=X_scores,
    weights=best_weights,
    unregistration_weeks=unreg_weeks,
    discount_factor=0.15,
)

print(f"Posteriors computed in {time.perf_counter() - t0:.2f}s, shape: {all_posteriors.shape}")

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
# OOF Posteriors for Meta-Classifier Training
print("Generating OOF Posteriors for Meta-Classifier...")
n_train = len(train_idx)
oof_posteriors = np.zeros((n_train, 40, 4))
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_groups = base_students_df["id_student"][train_idx].to_numpy()
X_c_tr, X_s_tr, y_tr = X_clicks[train_idx], X_scores[train_idx], targets[train_idx]
unreg_tr = unreg_weeks[train_idx]

for tr_idx, vl_idx in sgkf.split(X_c_tr, y_tr, train_groups):
    fold_nb = DualPathNaiveBayes(num_classes=4, var_smoothing=1e-4, discount_factor=0.15)
    fold_nb.fit(X_c_tr[tr_idx], X_s_tr[tr_idx], y_tr[tr_idx])
    oof_posteriors[vl_idx] = fold_nb.predict_weekly_posteriors(
        X_c_tr[vl_idx], X_s_tr[vl_idx], weights=best_weights,
        unregistration_weeks=unreg_tr[vl_idx], discount_factor=0.15
    )

# Enriched Stacking Meta-Classifier (LightGBM + Granular Features)
cutoffs = [4, 8, 12, 39]
class_names = ["Pass", "Withdrawn", "Fail", "Distinction"]
results_summary = []

print("=" * 78)
print("PHASE 2.2 ENRICHED MULTI-CUTOFF STACKING BENCHMARK (OOF + TEST EVAL)")
print("=" * 78)

for cutoff in cutoffs:
    # Cumulatives up to cutoff week
    clicks_sub = X_clicks[:, :cutoff + 1]
    scores_sub = X_scores[:, :cutoff + 1]
    
    sum_clicks = np.sum(clicks_sub, axis=1, keepdims=True)
    mean_clicks = np.mean(clicks_sub, axis=1, keepdims=True)
    active_weeks = np.sum(clicks_sub > 0, axis=1, keepdims=True).astype(np.float64)
    
    with np.errstate(invalid="ignore"):
        mean_score = np.nanmean(scores_sub, axis=1, keepdims=True)
    mean_score = np.nan_to_num(mean_score, nan=0.0)
    submission_count = np.sum(~np.isnan(scores_sub), axis=1, keepdims=True).astype(np.float64)
    
    cum_feats_list = [sum_clicks, mean_clicks, active_weeks, mean_score, submission_count]
    
    # Granular activities cumulative
    for act in activity_cols:
        act_sub = X_activities[act][:, :cutoff + 1]
        act_sum = np.sum(act_sub, axis=1, keepdims=True)
        cum_feats_list.append(act_sum)
        
    cum_feats = np.hstack(cum_feats_list)
    
    P_train_oof = oof_posteriors[:, cutoff, :]
    P_test = all_posteriors[test_idx, cutoff, :]
    
    X_train_stacked = stack_enriched_features(P_train_oof, cum_feats[train_idx], encoded_demographics[train_idx])
    X_test_stacked = stack_enriched_features(P_test, cum_feats[test_idx], encoded_demographics[test_idx])
    
    scaler = StandardScaler().fit(X_train_stacked)
    X_train_s = scaler.transform(X_train_stacked)
    X_test_s = scaler.transform(X_test_stacked)
    
    test_unreg = unreg_weeks[test_idx]
    active_mask = ~((test_unreg >= 0) & (test_unreg < cutoff))
    
    X_test_s_active = X_test_s[active_mask]
    y_test_active = targets[test_idx][active_mask]
    P_test_active = P_test[active_mask]
    
    # LightGBM Meta-learner
    meta = MetaClassifierWrapper(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)
    meta.fit(X_train_s, y_train)
    
    y_pred = meta.predict(X_test_s_active)
    y_proba = meta.predict_proba(X_test_s_active)
    
    macro_f1 = f1_score(y_test_active, y_pred, average="macro")
    weighted_f1 = f1_score(y_test_active, y_pred, average="weighted")
    try:
        ovr_auc = roc_auc_score(y_test_active, y_proba, multi_class="ovr")
    except ValueError:
        ovr_auc = np.nan
    
    y_at_risk = np.isin(y_test_active, [1, 2]).astype(int)
    pred_at_risk = ((y_proba[:, 1] + y_proba[:, 2]) >= 0.50).astype(int)
    at_risk_recall = recall_score(y_at_risk, pred_at_risk)
    
    bayesian_pred = np.argmax(P_test_active, axis=1)
    bayesian_macro_f1 = f1_score(y_test_active, bayesian_pred, average="macro")
    
    print(f"\n>>> Cutoff Week {cutoff:02d} (Active Test Students: {np.sum(active_mask)} / {len(test_idx)}) <<<")
    print(f"  Enriched Model: Macro F1: {macro_f1:.4f} | Weighted F1: {weighted_f1:.4f} | OvR AUC: {ovr_auc:.4f} | At-Risk Recall: {at_risk_recall:.4f}")
    
    results_summary.append({
        "cutoff_week": cutoff,
        "bayesian_macro_f1": bayesian_macro_f1,
        "enriched_macro_f1": macro_f1,
        "enriched_weighted_f1": weighted_f1,
        "enriched_ovr_auc": ovr_auc,
        "enriched_at_risk_recall": at_risk_recall,
    })

results_df = pd.DataFrame(results_summary)
print("\nSummary Table:")
print(results_df[["cutoff_week", "enriched_macro_f1", "enriched_ovr_auc", "enriched_at_risk_recall", "bayesian_macro_f1"]].to_string(index=False))


In [ ]:
# Visualizations: Plot Trajectories and Cutoff Performance
# 1. Macro F1, AUC, and At-Risk Recall progression across cutoffs
fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

ax1.plot(results_df["cutoff_week"], results_df["enriched_macro_f1"], "b-o", label="Enriched Macro F1", linewidth=2)
ax1.plot(results_df["cutoff_week"], results_df["enriched_at_risk_recall"], "g-^", label="At-Risk Recall", linewidth=2)
ax1.plot(results_df["cutoff_week"], results_df["bayesian_macro_f1"], "b--s", label="Bayesian Macro F1", linewidth=2)
ax2.plot(results_df["cutoff_week"], results_df["enriched_ovr_auc"], "r-o", label="Enriched ROC-AUC", linewidth=2)

ax1.set_xlabel("Course Cutoff Week", fontsize=12)
ax1.set_ylabel("Macro F1 / At-Risk Recall", color="b", fontsize=12)
ax2.set_ylabel("One-vs-Rest ROC-AUC", color="r", fontsize=12)
plt.title("AURA Enriched Early Warning Progression (Phase 2.1)", fontsize=14)
ax1.grid(True, linestyle="--", alpha=0.5)
fig.tight_layout()
plt.savefig(FIG_DIR / "early_warning_progression.png", dpi=150)
plt.close()
print("Saved early_warning_progression.png")

# 2. Plot Sample Student Risk Trajectories
sample_indices = [val_idx[0], val_idx[1], val_idx[2]]
fig, axes = plt.subplots(len(sample_indices), 1, figsize=(10, 3 * len(sample_indices)), sharex=True)

weeks = np.arange(40)
colors = ["#2ecc71", "#e74c3c", "#f39c12", "#3498db"]

for ax, s_idx in zip(axes, sample_indices):
    student_probs = all_posteriors[s_idx, :, :]  # (40, 4)
    true_label = class_names[targets[s_idx]]
    unreg = unreg_weeks[s_idx]
    
    for k_class in range(4):
        ax.plot(weeks, student_probs[:, k_class], label=class_names[k_class], color=colors[k_class], linewidth=2)
    
    if unreg >= 0:
        ax.axvline(x=unreg, color="black", linestyle=":", label=f"Unregistered (Wk {int(unreg)})")
    
    ax.set_ylabel("Posterior P(Y=k)", fontsize=10)
    student_id = student_ids[s_idx]
    ax.set_title(f"Student ID: {student_id} | True Outcome: {true_label}", fontsize=11)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.4)

axes[-1].set_xlabel("Course Timeline Week (t)", fontsize=12)
fig.tight_layout()
plt.savefig(FIG_DIR / "sample_student_trajectories.png", dpi=150)
plt.close()
print("Saved sample_student_trajectories.png")

In [ ]:
# Persist Trained Artifacts and Metrics
print("Persisting Phase 2.1 model metrics and calibrated weights...")
np.savez_compressed(
    PROCESSED_DIR / "phase2_model_metrics.npz",
    best_weights=best_weights,
    best_config=np.array([list(best_config.items())], dtype=object),
    results_summary=results_df.to_dict(orient="records"),
)
print("Phase 2.1 Model Training & Benchmark Pipeline Completed Successfully.")